# 🌸 Flower Distributed Federated Learning Pipeline via gRPC (FL-IoT-IDS)
## Decentralized Edge Intrusion Detection over TCP/IP gRPC Sockets
### Graduation Thesis: *Xây dựng và đánh giá prototype hệ thống phát hiện xâm nhập IoT sử dụng Federated Learning trong môi trường dữ liệu non-IID*
**Author:** Nguyễn Việt Kỳ Quân (23521267) | **Advisor:** PGS.TS Lê Trung Quân | **Institution:** VNU-HCM UIT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NVKQ2022/FedLearning/blob/main/notebooks/flower_federated_pipeline.ipynb)

**Repository:** `NVKQ2022/FedLearning` | **Transport:** Decoupled Edge Multi-Process via gRPC (TCP/IP Port 8080)


---
### 🛠️ Step 1: Environment Setup & gRPC Transport Verification
We verify GPU acceleration, install `flwr`, synchronize repository code, and enforce deterministic seeds across all random generators.

> [!NOTE]
> **Decoupled Edge Prototype Architecture (Proposal Section 5 & 7):**  
> The system operates as a distributed prototype consisting of:
> 1. **Central Federated Server Process (`flower_server.py`)**: Runs aggregation algorithms, coordinates communication rounds, and evaluates global holdout metrics.
> 2. **Edge Client Processes (`flower_client.py`)**: Independent OS processes training locally on private partitions, communicating strictly over **gRPC network sockets** (TCP/IP Port 8080).


In [ ]:
import os, sys, subprocess, warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Compute Device: {device} | PyTorch: {torch.__version__}')

# Colab setup
if os.path.exists('/content'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        pass
    repo_dir = '/content/FedLearning'
    if not os.path.exists(repo_dir):
        subprocess.run(['git', 'clone', 'https://github.com/NVKQ2022/FedLearning.git', repo_dir], check=True)
    os.chdir(repo_dir)
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
elif os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

try:
    import flwr as fl
    print(f'✅ Flower Version: {fl.__version__}')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-warn-conflicts', 'flwr>=1.13.0'], check=True)
    import flwr as fl
    print(f'✅ Flower installed: {fl.__version__}')

from src.utils.seed import set_seed
set_seed(42)
print('✅ Deterministic random seed locked at 42.')

---
### ⚙️ Step 2: Global Hyperparameter Configuration (Proposal Section 8)
Configure experimental parameters for scenarios **E2** to **E6**:
* **Number of Clients ($K$):** $K \in \{5, 7, 10\}$ edge IoT nodes (Proposal Section 8, Scenario E6).
* **Communication Rounds ($T$):** Total server-client synchronization rounds $t = 1, \dots, T$.
* **Client Local Epochs ($E$):** Local SGD training passes on private client partition per round.
* **Dirichlet Skew ($lpha$):** $lpha = 1.0$ (E3 - mild), $lpha = 0.5$ (E4 - moderate), $lpha = 0.1$ (E5 - severe non-IID).
* **FedProx Coefficient ($\mu$):** $\mu = 0.0$ for FedAvg (E2/E4); $\mu = 0.05$ for FedProx (E5).


In [ ]:
from src.utils.config import (
    Experiment,
    DataConfig,
    ModelConfig,
    LossConfig,
    OptimizerConfig,
    FederatedConfig,
    FedAvg,
    FedProx,
)

CONFIG = Experiment(
    experiment_name="flower_e2_fedavg_iid",
    seed=42,
    data=DataConfig(
        sample_size=100000,                 # 100k samples for fast execution
        test_size=0.2,                      # 20% holdout test set
        val_size=0.1,                       # 10% centralized server validation set
        scaler_type="robust",               # 'robust' (median/IQR)
        batch_size=64,                      # Client local batch size
    ),
    model=ModelConfig(
        hidden_dims=(128, 64),              # Lightweight edge MLP (~15k params)
        dropout_rate=0.2,
    ),
    loss=LossConfig(
        loss_type="focal_loss",             # MultiClassFocalLoss
        class_weight_strategy="balanced",   # Inverse frequency class weighting
        focal_gamma=2.0,
    ),
    optimizer=OptimizerConfig(
        optimizer_type="adamw",
        learning_rate=1e-3,
        weight_decay=1e-4,
        max_grad_norm=5.0,
    ),
    federated=FederatedConfig(
        algorithm=FedAvg(),                 # Algorithm inside: FedAvg() or FedProx(mu=0.05)
        num_clients=5,                      # Common FL parameter: edge clients K
        num_rounds=10,                      # Common FL parameter: server rounds T
        local_epochs=2,                     # Common FL parameter: local epochs E
        partition_type="iid",               # 'iid' or 'dirichlet'
        dirichlet_alpha=0.5,                # Dirichlet concentration parameter (0.5 moderate, 0.1 severe)
    ),
)

CONFIG.display()


---
### 💾 Step 3: Leak-Free Preprocessing & Non-IID Dirichlet Partitioning (Proposal Section 2 & 4)

#### 1. Inverse Class Frequency Weights (Class Imbalance Mitigation):
To counter severe class imbalance where majority DDoS traffic overwhelms minority intrusions (Web-based, Brute-force):
$$w_c = \frac{N}{C \cdot N_c}$$
where $N$ is total training samples, $C = 8$ attack classes, and $N_c$ is the sample count of class $c$.

#### 2. Dirichlet Non-IID Data Partitioning (Proposal Section 4.2):
For each attack class $c \in \{1, \dots, C\}$, draw client allocation proportions from a Dirichlet distribution:
$$\mathbf{p}_c = (p_{c,1}, p_{c,2}, \dots, p_{c,K}) \sim \text{Dirichlet}(\alpha \cdot \mathbf{1}_K)$$
$$f(\mathbf{p}_c; \alpha) = \frac{\Gamma(K\alpha)}{\Gamma(\alpha)^K} \prod_{k=1}^K p_{c,k}^{\alpha - 1}$$
* $\alpha \to \infty$: Converges to identical IID partitions.
* $\alpha = 1.0$: Moderate variance across clients (Scenario E3).
* $\alpha = 0.5$: Substantial heterogeneity (Scenario E4).
* $\alpha = 0.1$: Severe non-IID skew where clients hold only 1–2 dominant classes (Scenario E5).

#### 3. Partition Statistical Heterogeneity Metric (Shannon Entropy):
To quantify client label diversity, we compute Shannon Entropy for each client $k$:
$$H(k) = - \sum_{c=1}^C P(y=c \mid k) \log_2 P(y=c \mid k)$$
Higher $H(k)$ indicates balanced class representation; low $H(k)$ indicates high concentration/skew.


In [ ]:
from src.data.preprocess import load_and_preprocess_ciciot2023, compute_balanced_class_weights
from src.data.partitioner import (
    StratifiedIIDPartitioner,
    DirichletNonIIDPartitioner,
    create_client_dataloaders,
    summarize_client_partitions
)

# Locate dataset CSV
candidate_paths = [
    CONFIG.gdrive_csv,
    CONFIG.alt_gdrive_csv,
    CONFIG.target_csv_path,
    'datasets/CICIOT2023/merged_CICIOT2023_data.csv',
    'datasets/merged_CICIOT2023_data.csv',
]
csv_path = next((p for p in candidate_paths if os.path.exists(p) and os.path.getsize(p) > 10*1024*1024), None)
if not csv_path:
    csv_path = CONFIG.target_csv_path

print(f"📁 Loading and preprocessing from: {csv_path}")
data = load_and_preprocess_ciciot2023(
    csv_path=csv_path,
    test_size=CONFIG.test_size,
    val_size=CONFIG.val_size,
    sample_size=CONFIG.sample_size,
    scaler_type=CONFIG.scaler_type,
    batch_size=CONFIG.batch_size,
    random_state=CONFIG.seed
)

class_weights = compute_balanced_class_weights(data['y_train'], data['num_classes'], strategy=CONFIG.class_weight_strategy)
class_names = data['class_names']
minority_classes = ['Web-based', 'Brute-force']

# Generate client partitions
if CONFIG.partition_type == "iid":
    partitioner = StratifiedIIDPartitioner(num_clients=CONFIG.num_clients, seed=CONFIG.seed)
else:
    partitioner = DirichletNonIIDPartitioner(
        num_clients=CONFIG.num_clients,
        alpha=CONFIG.dirichlet_alpha,
        min_samples_per_client=100,
        seed=CONFIG.seed
    )

client_partitions = partitioner.partition(data['X_train'], data['y_train'])
client_loaders = create_client_dataloaders(data['X_train'], data['y_train'], client_partitions, batch_size=CONFIG.batch_size)

summary_df = summarize_client_partitions(client_partitions, data['y_train'], class_names=class_names)
display(summary_df[['client_id', 'total_samples', 'dominant_class', 'dominant_pct', 'entropy']])

---
### 🏛️ Step 4: Edge Neural Architecture & Server Evaluation Callback (Proposal Section 3)

#### 1. Edge MLP Architecture (`TabularIoTMLPModel` - Proposal Table 3):
A lightweight multi-layer perceptron designed for resource-constrained IoT devices ($d_{\text{in}} = 39$ flow features, footprint $\approx 54.28\text{ KB}$):
$$\mathbf{h}_1 = \text{Dropout}\big(\text{ReLU}(\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1)\big), \quad \mathbf{W}_1 \in \mathbb{R}^{128 \times 39}$$
$$\mathbf{h}_2 = \text{Dropout}\big(\text{ReLU}(\mathbf{W}_2 \mathbf{h}_1 + \mathbf{b}_2)\big), \quad \mathbf{W}_2 \in \mathbb{R}^{64 \times 128}$$
$$\hat{\mathbf{y}} = \text{softmax}(\mathbf{W}_3 \mathbf{h}_2 + \mathbf{b}_3), \quad \mathbf{W}_3 \in \mathbb{R}^{8 \times 64}$$

#### 2. Loss Function (Multi-Class Focal Loss):
Suppresses easy majority-class negative gradients to emphasize hard minority attack classes:
$$\mathcal{L}_{\text{Focal}}(p_t) = - \alpha_t (1 - p_t)^\gamma \log(p_t), \quad \gamma = 2.0$$


In [ ]:
from src.models.tabular_mlp import TabularIoTMLPModel
from src.losses.focal_loss import build_loss_function
from src.federated.flower_server import build_flower_server_eval_fn

global_model = TabularIoTMLPModel(
    input_dim=data['input_dim'],
    hidden_dims=CONFIG.hidden_dims,
    num_classes=data['num_classes'],
    dropout_rate=CONFIG.dropout_rate
).to(device)

criterion = build_loss_function(
    loss_type=CONFIG.loss_type,
    class_weights=class_weights if CONFIG.class_weight_strategy != 'none' else None,
    gamma=CONFIG.focal_gamma,
    device=device
)

model_size_bytes = sum(p.numel() * p.element_size() for p in global_model.parameters())

# Server-side holdout evaluation callback executed after every communication round
flower_eval_fn = build_flower_server_eval_fn(
    model=global_model,
    val_loader=data['val_loader'],
    class_names=class_names,
    criterion=criterion,
    minority_classes=minority_classes,
    device=device
)

print(f"✅ Global Model: {global_model.get_num_parameters():,} params ({model_size_bytes/1024:.2f} KB)")
print("✅ Server Holdout Evaluation Callback built successfully.")

---
### 📦 Step 5: Export Scenario Hierarchy & Client Partitions for gRPC
Before launching the gRPC processes, we persist client partitions, validation data, and configuration to `scenarios/<scenario_name>/`.
This ensures reproducible isolated file access for each client process $k \in \{0, \dots, K-1\}$.


In [ ]:
from src.federated.scenario import create_federated_scenario
import os

scenario_dir = f"scenarios/{CONFIG.experiment_name}"
print(f"📦 Generating and exporting scenario structure for gRPC: {scenario_dir}")

create_federated_scenario(
    scenario_name=CONFIG.experiment_name,
    client_partitions=client_partitions,
    X_train=data['X_train'],
    y_train=data['y_train'],
    X_val=data['X_val'],
    y_val=data['y_val'],
    class_names=class_names,
    config={
        "scenario_name": CONFIG.experiment_name,
        "strategy": CONFIG.federated_strategy,
        "mu": CONFIG.mu if CONFIG.federated_strategy == "fedprox" else 0.0,
        "num_clients": CONFIG.num_clients,
        "num_rounds": CONFIG.num_rounds,
        "local_epochs": CONFIG.local_epochs,
        "partition_type": CONFIG.partition_type,
        "dirichlet_alpha": CONFIG.dirichlet_alpha,
        "seed": CONFIG.seed,
    },
    base_dir="scenarios",
    generate_plots=True
)

print(f"✅ Scenario artifacts ready at: {scenario_dir}/")
print(f"   - Server Validation Data: {scenario_dir}/server/val_data.npz")
print(f"   - Client Partitions:      {scenario_dir}/client_*/partition.npz")

---
### 🌐 Step 6: Multi-Process Flower gRPC Execution & Mathematical Optimization (Proposal Section 6)

We launch the central Flower Server (`0.0.0.0:8080`) and $K$ Edge Clients as independent OS processes communicating over real **gRPC network sockets** (TCP/IP).

#### 1. Client Local Optimization with FedProx Proximal Penalty (Proposal Hình 6.2):
Each participating edge client $k$ optimizes its local objective with an optional proximal penalty constrained to global weights $\mathbf{w}_t$:
$$\min_{\mathbf{w}} h_k(\mathbf{w}; \mathbf{w}_t) = F_k(\mathbf{w}) + \frac{\mu}{2} \|\mathbf{w} - \mathbf{w}_t\|_2^2$$
where:
* $F_k(\mathbf{w}) = \frac{1}{n_k} \sum_{i=1}^{n_k} \mathcal{L}(f(\mathbf{x}_i; \mathbf{w}), y_i)$ is client $k$'s local empirical risk.
* $\mathbf{w}_t$ is the frozen global model received at the start of communication round $t$.
* $\mu \ge 0$ is the proximal regularization parameter ($\mu = 0.0$ reduces to standard FedAvg; $\mu = 0.05$ restricts non-IID client drift).
* $\|\mathbf{w} - \mathbf{w}_t\|_2^2 = \sum_{l=1}^L \|\mathbf{w}^{(l)} - \mathbf{w}_t^{(l)}\|_2^2$ is the Euclidean parameter distance across all neural layers.

#### 2. Server Aggregation via Sample-Weighted FedAvg (Proposal Hình 6.1):
After receiving model updates from participating clients, the server aggregates parameters proportionally to client dataset size:
$$\mathbf{w}_{t+1} = \sum_{k=1}^K \frac{n_k}{N} \mathbf{w}_k^{t+1}, \quad \text{where } N = \sum_{k=1}^K n_k$$


In [ ]:
import time
from src.federated.grpc_runner import run_flower_grpc

print(f"🌐 Starting Flower gRPC Federated Learning: {CONFIG.num_clients} clients, {CONFIG.num_rounds} rounds...")
print(f"Strategy: {CONFIG.federated_strategy.upper()} | mu: {CONFIG.mu if CONFIG.federated_strategy == 'fedprox' else 0.0} | Local Epochs: {CONFIG.local_epochs}")
print(f"Transport: Distributed gRPC network sockets (port 8080)")
print("=" * 80)

sim_start = time.perf_counter()

round_history = run_flower_grpc(
    scenario_name=CONFIG.experiment_name,
    num_clients=CONFIG.num_clients,
    rounds=CONFIG.num_rounds,
    strategy=CONFIG.federated_strategy,
    mu=CONFIG.mu if CONFIG.federated_strategy == 'fedprox' else 0.0,
    port=8080,
    scenarios_dir="scenarios",
    device="cpu",
    stream_logs=True,
    auto_find_port=True
)

sim_time = time.perf_counter() - sim_start
print("=" * 80)
print(f"🎉 Flower gRPC Training completed in {sim_time:.1f}s ({sim_time/60:.2f} min)!")
if round_history and "val_macro_f1" in round_history and len(round_history["val_macro_f1"]) > 0:
    best_f1 = max(round_history["val_macro_f1"])
    best_rnd = round_history["val_macro_f1"].index(best_f1) + 1
    print(f"Best Round: {best_rnd} | Best Val Macro-F1: {best_f1 * 100:.2f}%")

---
### 📈 Step 7: Convergence, Network Overhead & Parameter Drift (Proposal Section 9)

We evaluate system metrics defined in **Proposal Table 9**:

#### 1. Cumulative Communication Overhead Formula (Proposal Section 9):
The bidirectional network payload exchanged between server and clients up to round $t$ is computed as:
$$\text{CommCost}(t) = 2 \times \sum_{i=1}^t m_i \cdot B \quad [\text{Bytes / MB}]$$
where:
* $m_i$: number of participating clients in round $i$ ($m_i = K \cdot \text{fraction\_fit}$).
* $B$: neural model parameter footprint in bytes ($B \approx 55,584\text{ Bytes} \approx 54.28\text{ KB}$ for `TabularIoTMLPModel`).
* Factor $2$: accounts for downlink broadcast ($\text{Server} \to \text{Client}$) and uplink update transmission ($\text{Client} \to \text{Server}$).

#### 2. Client Parameter Drift Metric ($\Delta \mathbf{w}$):
Quantifies the divergence between local model updates and the global parameter vector caused by non-IID data skew:
$$\Delta \mathbf{w}_k^t = \|\mathbf{w}_{\text{local}, k}^t - \mathbf{w}_{\text{global}}^t\|_2 = \sqrt{\sum_{l=1}^L \|\mathbf{w}_{\text{local}, k}^{(l)} - \mathbf{w}_{\text{global}, t}^{(l)}\|_2^2}$$


In [ ]:
import matplotlib.pyplot as plt

rounds = round_history.get("round", [])

if len(rounds) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5), dpi=120)
    
    # Left: Macro-F1 Convergence
    val_f1 = round_history["val_macro_f1"][-len(rounds):]
    ax1.plot(rounds, [f * 100 for f in val_f1], 'o-', color='#2980b9', linewidth=2)
    ax1.set_title(f"Flower {CONFIG.federated_strategy.upper()} Macro-F1 Convergence (gRPC)", fontsize=11, fontweight='bold')
    ax1.set_xlabel("Round", fontsize=10)
    ax1.set_ylabel("Macro-F1 (%)", fontsize=10)
    ax1.grid(True, linestyle="--", alpha=0.5)
    
    # Right: Cumulative Communication Overhead (MB)
    comm_mb = round_history.get("comm_cost_mb", [0.0] * len(rounds))
    ax2.plot(rounds, comm_mb, 's-', color='#27ae60', linewidth=2)
    ax2.set_title("Cumulative Network Communication Overhead (MB)", fontsize=11, fontweight='bold')
    ax2.set_xlabel("Round", fontsize=10)
    ax2.set_ylabel("Transferred (MB)", fontsize=10)
    ax2.grid(True, linestyle="--", alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    print(f"📊 Total Payload Exchanged: {comm_mb[-1]:.2f} MB")

---
### 🏆 Step 8: Final Holdout Test Evaluation (Proposal Table 9)

The best checkpoint is evaluated on the stationary, unseen global test set $\mathcal{D}_{\text{test}}$.

#### 1. Macro-Averaged F1 Score (Primary Detection Metric):
Prevents high-volume attack classes (DDoS/DoS) from masking catastrophic degradation on minority classes:
$$\text{Macro-F1} = \frac{1}{C} \sum_{c=1}^C \text{F1}_c = \frac{1}{C} \sum_{c=1}^C \frac{2 \cdot P_c \cdot R_c}{P_c + R_c}$$
where $P_c = \frac{\text{TP}_c}{\text{TP}_c + \text{FP}_c}$ and $R_c = \frac{\text{TP}_c}{\text{TP}_c + \text{FN}_c}$.

#### 2. Minority Intrusion Recall (Stealth Infiltration Attacks):
Evaluates detection sensitivity on critical, low-frequency intrusion classes:
$$\text{Minority Recall} = \frac{\sum_{c \in \mathcal{C}_{\text{minority}}} \text{TP}_c}{\sum_{c \in \mathcal{C}_{\text{minority}}} (\text{TP}_c + \text{FN}_c)}, \quad \mathcal{C}_{\text{minority}} = \{\text{Web-based}, \text{Brute-force}\}$$

#### 3. Overall Accuracy & Confusion Matrix:
$$\text{Accuracy} = \frac{\sum_{c=1}^C \text{TP}_c}{N_{\text{test}}}$$
$$\mathbf{CM}_{i, j} = \big| \{ (\mathbf{x}, y) \in \mathcal{D}_{\text{test}} \mid y = i \text{ and } \hat{y} = j \} \big|$$


In [ ]:
from src.evaluation.evaluator import evaluate_comprehensive
from src.visualization import plot_confusion_matrix

# Restore best global weights from scenario server checkpoint
best_ckpt = f"scenarios/{CONFIG.experiment_name}/server/best_weights.npz"
if not os.path.exists(best_ckpt):
    # Fallback to checkpoints directory
    best_ckpt = f"checkpoints/flower_{CONFIG.federated_strategy}_best_weights.npz"

if os.path.exists(best_ckpt):
    npz = np.load(best_ckpt)
    best_weights = [npz[f"arr_{i}"] for i in range(len(npz.files))]
    global_model.set_weights(best_weights)
    print(f"✅ Restored best weights from: {best_ckpt}")

test_results = evaluate_comprehensive(
    model=global_model,
    dataloader=data['test_loader'],
    class_names=class_names,
    minority_classes=minority_classes,
    device=device
)

print('=' * 65)
print(f"🏆 FLOWER {CONFIG.federated_strategy.upper()} FINAL HOLDOUT TEST EVALUATION (gRPC)")
print('=' * 65)
print(f"Test Accuracy:    {test_results['accuracy'] * 100:.2f}%")
print(f"Macro-F1 Score:   {test_results['macro_f1'] * 100:.2f}%")
print(f"Weighted-F1:      {test_results['weighted_f1'] * 100:.2f}%")
print(f"Macro-Precision:  {test_results['macro_precision'] * 100:.2f}%")
print(f"Macro-Recall:     {test_results['macro_recall'] * 100:.2f}%")
print('-' * 65)
print('Minority Class Recall (Stealth Infiltration Intrusions):')
for cls, rec in test_results['minority_recall'].items():
    print(f"   • {cls:<15}: {rec * 100:.2f}%")
print('=' * 65)

# Render Confusion Matrix
plot_confusion_matrix(
    cm=test_results['confusion_matrix'],
    class_names=class_names,
    title=f"Flower {CONFIG.federated_strategy.upper()} (gRPC): Confusion Matrix",
    show=True
)

---
### 💾 Step 9: Export Scenario Results & Scientific Artifacts
Persists evaluation metrics, convergence history, and holdout performance matrices into `scenarios/<scenario_name>/server/metrics.json` for thesis reporting and cross-scenario ablation comparisons.


In [ ]:
import json

scenario_dir = f"scenarios/{CONFIG.experiment_name}"
metrics_export = {k: v for k, v in test_results.items() if k != 'confusion_matrix'}
metrics_export['round_history'] = round_history
metrics_path = f"{scenario_dir}/server/metrics.json"

with open(metrics_path, 'w') as f:
    json.dump(metrics_export, f, indent=2)

print(f"✅ Successfully updated scenario metrics: {metrics_path}")
print(f"\n🎉 Flower {CONFIG.federated_strategy.upper()} gRPC Pipeline complete!")
